# Compare TotalSegmentator vs DentalSegmentator on one CT/CBCT

This notebook runs both adapters from the Facial Align repo on a single scan.

Important note:
- `DentalSegmentatorModel` is **not** using the Zenodo dental weights yet.
- In the current repo, it **falls back to TotalSegmentator's `teeth` subtask**.
- So this notebook is useful for verifying the current pipeline and proving whether the outputs are effectively the same.


In [ ]:
# Install dependencies if needed
# Uncomment if this runtime is fresh.
# %pip install -q totalsegmentator SimpleITK scipy numpy matplotlib pandas nibabel pydicom


In [ ]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Try to locate repo root automatically from the notebook location.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "services").exists():
    repo_root = repo_root.parent

if not (repo_root / "services").exists():
    raise RuntimeError("Could not locate repo root containing the 'services' directory.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)


In [ ]:
import SimpleITK as sitk

def load_volume(path):
    path = Path(path)
    if path.is_dir():
        # DICOM series
        reader = sitk.ImageSeriesReader()
        series_ids = reader.GetGDCMSeriesIDs(str(path))
        if not series_ids:
            raise ValueError(f"No DICOM series found in {path}")
        series_files = reader.GetGDCMSeriesFileNames(str(path), series_ids[0])
        reader.SetFileNames(series_files)
        image = reader.Execute()
    else:
        # NIfTI / NRRD / MHA / others supported by SimpleITK
        image = sitk.ReadImage(str(path))

    volume = sitk.GetArrayFromImage(image)  # (Z, Y, X)

    spacing_xyz = image.GetSpacing()        # (X, Y, Z)
    direction = image.GetDirection()
    origin = image.GetOrigin()

    return {
        "image": image,
        "volume": volume,
        "spacing_xyz": spacing_xyz,
        "origin": origin,
        "direction": direction,
        "size_xyz": image.GetSize(),
    }

# Set this to your CT/CBCT file or DICOM folder
SCAN_PATH = Path("PUT_YOUR_SCAN_PATH_HERE")

scan = load_volume(SCAN_PATH)
volume = scan["volume"]
spacing_xyz = tuple(float(x) for x in scan["spacing_xyz"])

print("Loaded shape (Z,Y,X):", volume.shape)
print("Spacing (X,Y,Z):", spacing_xyz)
print("Intensity range:", float(volume.min()), "to", float(volume.max()))


In [ ]:
from services.inference.model_registry import ModelVersion, ModelType
from services.inference.adapters.totalsegmentator_adapter import TotalSegmentatorModel
from services.inference.adapters.dental_adapter import DentalSegmentatorModel

device = "gpu"  # change to "cpu" if needed by your runtime / TotalSegmentator setup

ts_version = ModelVersion(
    name="totalsegmentator",
    version="2.5.0",
    model_type=ModelType.SEGMENTATION,
    architecture="nnU-Net",
)

dental_version = ModelVersion(
    name="dental_segmentator",
    version="phase1-fallback",
    model_type=ModelType.DENTAL_SEGMENTATION,
    architecture="nnU-Net",
)

ts_model = TotalSegmentatorModel(ts_version, device=device)
dental_model = DentalSegmentatorModel(dental_version, device=device)

# In the current repo, load() just marks the dental adapter ready and still falls back to TotalSegmentator.
dental_model.load(weights_path="phase1-fallback-no-zenodo-weights")

print("TotalSegmentator loaded:", ts_model.is_loaded)
print("DentalSegmentator loaded:", dental_model.is_loaded)


In [ ]:
# Run TotalSegmentator on craniofacial structures
ts_cmf = ts_model.predict(
    volume,
    spacing=spacing_xyz,
    subtask="craniofacial_structures",
    fast=False,
)

print("TotalSegmentator craniofacial structures:")
print("Structures:", sorted(ts_cmf["masks"].keys()))
print("Inference time (ms):", ts_cmf["inference_time_ms"])


In [ ]:
# Run TotalSegmentator on teeth directly
ts_teeth = ts_model.predict(
    volume,
    spacing=spacing_xyz,
    subtask="teeth",
    fast=False,
)

print("TotalSegmentator teeth:")
print("Tooth structures:", sorted(ts_teeth["masks"].keys())[:10], "...")
print("Num tooth masks:", len(ts_teeth["masks"]))
print("Inference time (ms):", ts_teeth["inference_time_ms"])


In [ ]:
# Run the DentalSegmentator adapter
dental_result = dental_model.predict(
    volume,
    spacing=spacing_xyz,
)

print("DentalSegmentator output:")
print("Teeth present:", dental_result["teeth_present"])
print("Num tooth masks:", len(dental_result["tooth_masks"]))
print("Inference time (ms):", dental_result["inference_time_ms"])


In [ ]:
# Compare TotalSegmentator teeth output vs DentalSegmentator wrapper output

def dice(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    denom = a.sum() + b.sum()
    return 1.0 if denom == 0 else 2.0 * inter / denom

rows = []
all_fdi = sorted(dental_result["teeth_present"])

for fdi in all_fdi:
    ts_key = f"tooth_{fdi}"
    dental_key = f"FDI-{fdi}"
    if ts_key in ts_teeth["masks"] and dental_key in dental_result["tooth_masks"]:
        d = dice(ts_teeth["masks"][ts_key], dental_result["tooth_masks"][dental_key])
        rows.append({
            "FDI": fdi,
            "dice_ts_vs_dental_wrapper": d,
            "ts_confidence": ts_teeth["confidences"].get(ts_key),
            "dental_confidence": dental_result["confidences"].get(dental_key),
            "ts_voxels": int(ts_teeth["masks"][ts_key].sum()),
            "dental_voxels": int(dental_result["tooth_masks"][dental_key].sum()),
        })

comparison_df = pd.DataFrame(rows).sort_values("FDI").reset_index(drop=True)
comparison_df


In [ ]:
# Expected result for the current repo:
# the per-tooth Dice should be 1.0 or extremely close, because DentalSegmentator
# currently wraps TotalSegmentator's teeth task.

if len(comparison_df):
    print("Mean Dice:", comparison_df["dice_ts_vs_dental_wrapper"].mean())
    print("Min Dice:", comparison_df["dice_ts_vs_dental_wrapper"].min())
else:
    print("No overlapping teeth found to compare.")


In [ ]:
# Visualize one slice with a simple overlay

def pick_middle_nonzero_slice(mask_3d):
    z_sums = mask_3d.reshape(mask_3d.shape[0], -1).sum(axis=1)
    nonzero = np.where(z_sums > 0)[0]
    if len(nonzero) == 0:
        return mask_3d.shape[0] // 2
    return int(nonzero[len(nonzero) // 2])

# Choose a structure to inspect
inspect_key = "mandible"
if inspect_key not in ts_cmf["masks"]:
    inspect_key = next(iter(ts_cmf["masks"]))

mask = ts_cmf["masks"][inspect_key]
z = pick_middle_nonzero_slice(mask)

plt.figure(figsize=(8, 8))
plt.imshow(volume[z], cmap="gray")
plt.imshow(np.ma.masked_where(mask[z] == 0, mask[z]), alpha=0.4)
plt.title(f"{inspect_key} overlay on slice z={z}")
plt.axis("off")
plt.show()


In [ ]:
# Save outputs if you want to inspect them in 3D Slicer

output_dir = repo_root / "notebook_outputs" / "segmentation_compare"
output_dir.mkdir(parents=True, exist_ok=True)

def save_mask_like_reference(mask_array, reference_image, out_path):
    img = sitk.GetImageFromArray(mask_array.astype(np.uint8))
    img.SetSpacing(reference_image.GetSpacing())
    img.SetOrigin(reference_image.GetOrigin())
    img.SetDirection(reference_image.GetDirection())
    sitk.WriteImage(img, str(out_path))

# Save combined masks
save_mask_like_reference(ts_cmf["combined_mask"], scan["image"], output_dir / "ts_cmf_combined.nii.gz")
save_mask_like_reference(ts_teeth["combined_mask"], scan["image"], output_dir / "ts_teeth_combined.nii.gz")
save_mask_like_reference(dental_result["dental_arch_upper"], scan["image"], output_dir / "dental_upper_arch.nii.gz")
save_mask_like_reference(dental_result["dental_arch_lower"], scan["image"], output_dir / "dental_lower_arch.nii.gz")

comparison_df.to_csv(output_dir / "tooth_comparison.csv", index=False)

print("Saved outputs to:", output_dir)
